In [293]:
import pandas as pd

In [294]:
sales_raw = pd.read_csv("sales_messy.csv")

In [295]:
customers  = pd.read_csv("customers.csv")

1)Load & profile the raw data — show

In [296]:
print("Shape:", sales_raw.shape)

Shape: (208, 9)


In [297]:
sales_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    object 
 2   customer_id  200 non-null    float64
 3   country      208 non-null    object 
 4   category     208 non-null    object 
 5   product      208 non-null    object 
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), object(4)
memory usage: 14.8+ KB


In [298]:
print("Missing Values per column:", sales_raw.isnull().sum())

Missing Values per column: order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64


In [299]:
print("Duplicate rows:", sales_raw.duplicated().sum())

Duplicate rows: 8


In [300]:
print("Country Unique:" , sales_raw["country"].unique())

Country Unique: ['GERMANY' 'Germany' 'France' ' France' 'Kazakhstan' 'UK' ' kazakhstan '
 'uk ' 'Poland' 'usa' 'USA' 'Russia']


Problems:The raw data has a few problems that should be fixed before starting the analysis. First, there are 8 duplicate rows, meaning the same record appears more than once. These duplicates should be removed. The country column is not consistent because some country names use different letter cases or contain extra spaces

2.Clean the Data

1)Removing Duplicates

In [301]:
sales = sales_raw.copy()

In [302]:
sales = sales.drop_duplicates() 

In [303]:
print("After dedup:", sales.shape)

After dedup: (200, 9)


2)Normalize Countries

In [304]:
sales["country"] = sales["country"].str.strip().str.title()

In [305]:
print(sales["country"].unique())

['Germany' 'France' 'Kazakhstan' 'Uk' 'Poland' 'Usa' 'Russia']


3)Filling missing discount with 0

In [306]:
sales["discount"] = sales["discount"].fillna(0)

4)Fill missing unit price with the median

In [307]:
median_price = sales["unit_price"].median()

In [308]:
print("Median unit_price used for fill:", median_price)

Median unit_price used for fill: 329.0


In [309]:
sales["unit_price"] = sales["unit_price"].fillna(median_price)

5)Drop rows with no customer_id

In [310]:
sales = sales.dropna(subset=["customer_id"])

In [311]:
sales.loc[:, "customer_id"] = sales["customer_id"].astype(int)

In [312]:
print("After dropping null customer_id:", sales.shape)

After dropping null customer_id: (193, 9)


6)Parse order date to datetime

In [313]:
sales.loc[:, "order_date"] = pd.to_datetime(sales["order_date"])

Verify missing values

In [314]:
critical_cols = ["customer_id", "order_date", "unit_price", "discount"]

In [315]:
print("missing values in critical columns:", sales[critical_cols].isnull().sum())

missing values in critical columns: customer_id    0
order_date     0
unit_price     0
discount       0
dtype: int64


Problems: The raw data contains missing discounts and unit prices, preventing accurate financial calculations.
Missing customer IDs make some transactions untraceable,
and the order_date is stored as text, which blocks time-based analysis.

3.Enrich — add a revenue

In [316]:
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

In [317]:
sales["order_date"] = pd.to_datetime(sales["order_date"])

In [318]:
sales["month"] = sales["order_date"].dt.month

In [319]:
sales[["order_id", "quantity", "unit_price", "discount", "revenue", "month"]].head()

,order_id,quantity,unit_price,discount,revenue,month
1,1002,3,59.99,0.10,161.973,7
2,1003,1,799.00,0.05,759.050,2
3,1004,4,899.00,0.20,2876.800,12
4,1005,5,549.00,0.10,2470.500,8
5,1006,6,59.99,0.15,305.949,2


Problems: The dataset lacks a direct financial metric for performance. Without calculating net revenue (accounting for discounts) and extracting the month, you cannot track actual cash flow or seasonal trends.

4.Merge — left-join customers.csv on customer_id

In [320]:
rows_before = len(sales)

In [321]:
sales = sales.merge(
    customers[["customer_id", "customer_name", "segment"]].drop_duplicates(subset=["customer_id"]),
    on="customer_id", 
    how="left"
)

In [322]:
rows_after = len(sales)

In [323]:
print(f"Rows before merge: {rows_before}")

Rows before merge: 193


In [324]:
print(f"Rows after  merge: {rows_after}")

Rows after  merge: 193


In [325]:
print("✓ Row count unchanged.")

✓ Row count unchanged.


In [326]:
sales.head()

,order_id,order_date,customer_id,country,category,product,quantity,unit_price,discount,revenue,month,customer_name,segment
0,1002,2025-07-24,28.0,Germany,Accessories,Webcam HD,3,59.99,0.10,161.973,7,Customer 28,Consumer
1,1003,2025-02-16,33.0,France,Phones,Pixela 8,1,799.00,0.05,759.050,2,Customer 33,Business
2,1004,2025-12-15,11.0,France,Laptops,ProBook 15,4,899.00,0.20,2876.800,12,Customer 11,Business
3,1005,2025-08-28,25.0,Kazakhstan,Laptops,EduBook 13,5,549.00,0.10,2470.500,8,Customer 25,Consumer
4,1006,2025-02-28,1.0,Uk,Accessories,Webcam HD,6,59.99,0.15,305.949,2,Customer 01,Consumer


Problems: Transaction data is isolated from customer context. Without joining the customer profiles, you cannot analyze sales by customer name or segment, and uncleaned lookup data risks creating duplicate rows during the merge.

5.Aggregations

1)Revenue per category

In [327]:
total_revenue = sales["revenue"].sum()

In [328]:
rev_category = (
    sales.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"revenue": "total_revenue"})
)

In [329]:
rev_category["share_%"] = (rev_category["total_revenue"] / total_revenue * 100).round(2)

In [330]:
rev_category["total_revenue"] = rev_category["total_revenue"].round(2)

In [331]:
rev_category

,category,total_revenue,share_%
0,Laptops,161187.40,54.97
1,Phones,63403.40,21.62
2,Monitors,58295.55,19.88
3,Accessories,10323.55,3.52


2)Revenue per Month

In [332]:
rev_month = (
    sales.groupby("month")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"revenue": "total_revenue"})
)

In [333]:
rev_month["total_revenue"] = rev_month["total_revenue"].round(2)

In [334]:
rev_month

,month,total_revenue
0,7,42529.43
1,10,33697.75
2,8,30827.43
3,4,26456.24
4,6,24754.84
5,5,23633.51
6,12,22739.75
7,3,19836.59
8,2,19631.08
9,11,19117.39


3)Revenue per customer segment

In [335]:
print("segment" in sales.columns)

True


In [336]:
rev_segment = (
    sales.groupby("segment")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"revenue": "total_revenue"})
)

In [337]:
print("segment" in sales.columns)

True


In [338]:
rev_segment["share_%"] = (rev_segment["total_revenue"] / total_revenue * 100).round(2)

In [339]:
rev_segment["total_revenue"] = rev_segment["total_revenue"].round(2)

In [340]:
rev_segment

,segment,total_revenue,share_%
0,Consumer,174850.84,59.63
1,Education,77782.03,26.53
2,Business,40577.03,13.84


Problems: Granular, row-by-row data is too dense for decision-making. Without grouping the data, you cannot identify your top-performing product categories, peak sales months, or dominant customer segments.

6.Conclusions

Pull scalar values for the conclusions narrative

In [341]:
top_cat = rev_category.iloc[0]

In [342]:
top_month = rev_month.iloc[0]

In [ ]:
top_segment = rev_segment.iloc[0]

In [ ]:
bottom_cat  = rev_category.iloc[-1]

In [ ]:
print("Top category:", top_cat['category'], top_cat['total_revenue'], top_cat['share_%'])

In [262]:
print("Best month:", top_month['month'], top_month['total_revenue'])


Best month: 7.0 42529.43


In [263]:
print("Top segment:", top_segment['segment'], top_segment['total_revenue'], top_segment['share_%'])

Top segment: Consumer 174850.84 59.63


In [264]:
print("Smallest cat:", bottom_cat['category'], bottom_cat['total_revenue'], bottom_cat['share_%'])

Smallest cat: Accessories 10323.55 3.52


In [265]:
print("Total revenue:", total_revenue)

Total revenue: 293209.89650000003
